# Example 04 — Two-DOF Tanh Friction: Nonlinear Modal Analysis

Backbone curve and forced FRF overlay for a 2-DOF chain with tanh dry-friction nonlinearity at DOF 0, demonstrating Nonlinear Modal Analysis (NMA) via amplitude continuation. (Krack & Gross 2019, §3)

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import scipy.optimize as sopt

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.nonlinearities.elements import tanh_dry_friction
from nlvib.systems.oscillators import ChainOfOscillators
from nlvib.solvers.harmonic_balance import hb_residual, hb_residual_nma
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions
from nlvib.visualization.plots import plot_backbone

In [ ]:
# --- System setup ---
MASSES = [1.0, 1.0]
STIFFNESSES = [1.0, 0.5, 0.0]  # open right end
DAMPINGS = [0.0, 0.0, 0.0]     # no linear damping for NMA
FRICTION_F0 = 0.3
FRICTION_C = 10.0
FRICTION_DOF = 0
N_HARMONICS = 5
FORCE_AMPLITUDE = 0.05
FORCE_DOF = 0

system = ChainOfOscillators(masses=MASSES, stiffnesses=STIFFNESSES, dampings=DAMPINGS)
system.add_nonlinear_element(tanh_dry_friction(f0=FRICTION_F0, c=FRICTION_C, dof_index=FRICTION_DOF))
print(f'System: {system!r}')

# Linear natural frequencies
M_dense = system.M.toarray()
K_dense = system.K.toarray()
evals, evecs = np.linalg.eig(np.linalg.solve(M_dense, K_dense))
idx = np.argsort(evals.real)
omega_lin = np.sqrt(np.maximum(evals.real[idx], 0.0))
mode_shapes = evecs[:, idx]
print(f'omega_1 = {omega_lin[0]:.4f} rad/s')
print(f'omega_2 = {omega_lin[1]:.4f} rad/s')
mode1 = mode_shapes[:, 0].real
if abs(mode1[0]) > 1e-12: mode1 = mode1 / mode1[0]
print(f'Mode 1 shape (norm. to DOF0=1): {mode1}')

In [ ]:
# --- NMA backbone via amplitude-fixed Newton sweeps ---
n_dof = system.n_dof
n_total = n_dof * (2 * N_HARMONICS + 1)
n_aug = n_total + 1
sin_h1_dof0_idx = 2 * n_dof
cos_h1_dof0_idx = n_dof

free_z_cols = np.array([i for i in range(n_aug) if i != sin_h1_dof0_idx], dtype=np.intp)
used_rows = free_z_cols.copy()

def solve_nma_at_amplitude(amplitude, omega_guess, Q_prev=None):
    if Q_prev is not None:
        z_full = np.append(Q_prev, omega_guess)
    else:
        z_full = np.zeros(n_aug)
        for i_dof in range(n_dof):
            z_full[sin_h1_dof0_idx + i_dof] = amplitude * mode1[i_dof]
        z_full[-1] = omega_guess
    z_full[sin_h1_dof0_idx] = amplitude
    z_full[cos_h1_dof0_idx] = 0.0
    z_free = z_full[free_z_cols].copy()

    def _full(z_f):
        z = np.zeros(n_aug); z[free_z_cols] = z_f; z[sin_h1_dof0_idx] = amplitude; return z
    def _res(z_f):
        z = _full(z_f); R, _ = hb_residual_nma(z, system, N_HARMONICS); return R[used_rows]

    z_f = z_free.copy()
    for _ in range(50):
        R_it = _res(z_f)
        if float(np.linalg.norm(R_it)) < 1e-8: break
        h_fd = float(np.sqrt(np.finfo(float).eps))
        J_fd = np.zeros((len(used_rows), len(z_f)))
        for jj in range(len(z_f)):
            h_j = max(h_fd * abs(z_f[jj]), h_fd)
            zp, zm = z_f.copy(), z_f.copy(); zp[jj] += h_j; zm[jj] -= h_j
            J_fd[:, jj] = (_res(zp) - _res(zm)) / (2*h_j)
        try: dz = np.linalg.solve(J_fd, -R_it)
        except np.linalg.LinAlgError: dz = np.linalg.lstsq(J_fd, -R_it, rcond=None)[0]
        alpha = 1.0
        for _ in range(10):
            if float(np.linalg.norm(_res(z_f + alpha*dz))) < float(np.linalg.norm(R_it)): break
            alpha *= 0.5
        z_f = z_f + alpha*dz

    if float(np.linalg.norm(_res(z_f))) > 1e-5: return None
    z_sol = _full(z_f)
    return z_sol[:n_total], float(z_sol[-1])

amplitude_levels = np.concatenate([np.geomspace(1e-3, 0.3, 30), np.linspace(0.3, 2.0, 20)])
amplitudes_bb, omegas_bb, Q_solutions = [], [], []
Q_prev, omega_guess = None, float(omega_lin[0])

print(f'NMA backbone: {len(amplitude_levels)} amplitude levels...')
for i_amp, amp in enumerate(amplitude_levels):
    res = solve_nma_at_amplitude(amp, omega_guess, Q_prev)
    if res is None: continue
    Q_sol, omega_sol = res
    if omega_sol <= 0.1 or omega_sol > 3.0: continue
    amplitudes_bb.append(amp); omegas_bb.append(omega_sol); Q_solutions.append(Q_sol.copy())
    Q_prev = Q_sol.copy(); omega_guess = omega_sol
    if (i_amp+1) % 10 == 0: print(f'  [amp={amp:.4e}] omega = {omega_sol:.4f} rad/s')

amplitudes_bb = np.array(amplitudes_bb)
omegas_bb = np.array(omegas_bb)
print(f'\nBackbone: {len(amplitudes_bb)} converged points')

In [ ]:
# --- Forced HB FRF sweep ---
excitation = {'dof': FORCE_DOF, 'amplitude': FORCE_AMPLITUDE, 'harmonic': 1}
omega_sweep = np.linspace(0.3, 1.2, 120)
amp_frf = np.zeros(len(omega_sweep))
Q_prev_frf = np.zeros(n_total)

for i_om, om in enumerate(omega_sweep):
    def _res(Q_vec, _om=om):
        R, _ = hb_residual(Q_vec, _om, system, N_HARMONICS, excitation); return R
    try:
        Q_sol = sopt.fsolve(_res, Q_prev_frf, full_output=False, xtol=1e-8)
        if np.linalg.norm(_res(Q_sol)) < 1e-4: Q_prev_frf = Q_sol.copy()
        else: Q_sol = Q_prev_frf.copy()
    except Exception: Q_sol = Q_prev_frf.copy()
    amp_frf[i_om] = np.sqrt(Q_sol[n_dof + FORCE_DOF]**2 + Q_sol[2*n_dof + FORCE_DOF]**2)

peak_idx_frf = int(np.argmax(amp_frf))
print(f'FRF peak: {amp_frf[peak_idx_frf]:.4e} at omega = {omega_sweep[peak_idx_frf]:.4f} rad/s')

In [ ]:
# --- Save plots ---
output_dir = Path('..') / 'examples' / '04_two_dof_tanh_friction' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

class BackboneResult:
    def __init__(self, omega, amplitude, stability):
        self.omega = omega; self.amplitude = amplitude; self.stability = stability

if len(amplitudes_bb) > 1:
    bb_result = BackboneResult(omega=omegas_bb, amplitude=amplitudes_bb,
                               stability=np.ones(len(amplitudes_bb), dtype=bool))
    fig_bb = plot_backbone(bb_result)
else:
    fig_bb, ax_bb = plt.subplots()
    ax_bb.set_xlabel('Modal amplitude'); ax_bb.set_ylabel(r'$\omega_n$ (rad/s)')
    ax_bb.set_title('Backbone (insufficient data)')
fig_bb.savefig(output_dir / 'backbone.png', dpi=150, bbox_inches='tight')
print('Saved backbone.png')

fig_overlay, ax_overlay = plt.subplots(figsize=(8, 5))
if len(amplitudes_bb) > 1:
    ax_overlay.plot(amplitudes_bb, omegas_bb, 'b-', linewidth=2.0, label='Backbone (NMA)')
ax_overlay.plot(amp_frf, omega_sweep, 'r--', linewidth=1.5, label=f'Forced FRF (F = {FORCE_AMPLITUDE})')
ax_overlay.set_xlabel('Amplitude'); ax_overlay.set_ylabel(r'Frequency $\omega$ (rad/s)')
ax_overlay.set_title(f'Backbone Curve with Forced FRF Overlay\n2-DOF tanh friction  (F = {FORCE_AMPLITUDE})')
ax_overlay.legend(loc='upper right'); ax_overlay.grid(True, linestyle='--', linewidth=0.4, alpha=0.6)
fig_overlay.savefig(output_dir / 'frf_overlay.png', dpi=150, bbox_inches='tight')
print('Saved frf_overlay.png')

In [ ]:
# --- Display backbone inline ---
fig_bb

In [ ]:
# --- Display FRF overlay inline ---
fig_overlay

In [ ]:
# --- Summary ---
print('=' * 60)
print('SUMMARY — Example 04: Two-DOF tanh friction NMA')
print('=' * 60)
print(f'  Linear omega_1:       {omega_lin[0]:.4f} rad/s')
print(f'  Linear omega_2:       {omega_lin[1]:.4f} rad/s')
print(f'  Backbone points:      {len(amplitudes_bb)}')
if len(amplitudes_bb) > 0:
    print(f'  Backbone freq range:  [{omegas_bb.min():.4f}, {omegas_bb.max():.4f}] rad/s')
    print(f'  Modal amp range:      [{amplitudes_bb.min():.4e}, {amplitudes_bb.max():.4e}]')
print(f'  FRF peak amplitude:   {amp_frf[peak_idx_frf]:.4e} at omega = {omega_sweep[peak_idx_frf]:.4f} rad/s')
print('=' * 60)